# LoRA Fine-Tuning Lab — structured extraction

Fine-tunes a small instruct model to turn a free-text support ticket into one
strict JSON object, and compares it honestly against two prompting baselines.

**Runs top to bottom on a free Colab T4.** The same code runs locally on Apple
MPS or CPU — `src/` is the source of truth and this notebook calls into it, so
there is no second copy of the training loop to drift.

### What is being measured

| Condition | What it is |
|---|---|
| `base_zeroshot` | the instruction only — the floor |
| `base_fewshot` | the same instruction + 4 worked examples — **the honest baseline** |
| `lora` | the adapter, no examples in the prompt |

Two held-out splits: `test_seen` (phrasing templates present in training) and
`test_unseen` (**template families the model never saw** — the generalisation
number worth quoting).

> Set the runtime to **T4 GPU** before running: Runtime → Change runtime type → T4 GPU.

## 1. Setup

In [ ]:
import os, subprocess, sys, textwrap

IN_COLAB = "google.colab" in sys.modules
print("colab:", IN_COLAB)

if IN_COLAB:
    !git clone -q https://github.com/jayu1023/lora-finetune-lab.git
    %cd lora-finetune-lab
    !pip install -q "transformers>=4.44" "peft>=0.12" accelerate datasets
else:
    # Local: assumes you are in the repo root with the venv installed.
    print("running locally — using the existing environment")

import torch
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

## 2. Build the dataset

Synthetic, from phrasing templates, and **deterministic** given the seed.

Three of the four fields are extraction — the signal is in the text. The fourth,
`escalate`, is *never stated*: it has to be inferred from the severity and the
category. Keeping both kinds in one schema is what lets the eval separate
"learned to emit JSON" from "learned the task".

The cell also prints the **majority-class baseline** per field. `escalate` is
~73% one class, so a model that always answers `false` already looks good on it —
which is exactly why that number has to be on screen next to the results.

In [ ]:
!python src/make_dataset.py

In [ ]:
import json
rows = [json.loads(l) for l in open("data/train.jsonl")]
print(f"{len(rows)} training examples. Two samples:\n")
for r in rows[:2]:
    print("TICKET:", r["ticket"].replace("\n", " ")[:150])
    print("LABEL :", json.dumps(r["label"]))
    print("FAMILY:", r["template_family"], "\n")

unseen = [json.loads(l) for l in open("data/test_unseen.jsonl")]
print("a held-out (unseen template family) example:")
print("TICKET:", unseen[0]["ticket"][:200])
print("FAMILY:", unseen[0]["template_family"])

## 3. Measure the baselines *before* training

Running the baselines first is deliberate. It is very easy to train something,
see a good number, and never find out that four examples in the prompt would
have got you most of the way — at which point the fine-tune has to justify
itself on cost and latency rather than on accuracy.

On a T4 this takes a few minutes. Use `--limit 40` for a faster pass.

In [ ]:
!python src/evaluate.py --conditions base_zeroshot base_fewshot --out evals/baselines.json

## 4. Train the adapter

The hyperparameters and what each one costs are in the README; the short version:

- **rank 16, alpha 32** — alpha/rank = 2, a scaling factor of 2 on the update.
- **target modules**: all four attention projections *plus* the three MLP
  projections. Attention-only (`q,v`) is the cheaper classic choice; the MLP is
  where token-level *formatting* behaviour lives, and this task changes the
  output format, so it is included.
- **lr 2e-4** — roughly 10x a full-fine-tune learning rate, which is normal for
  LoRA: only the adapter moves, and it starts from zero.
- **loss on the completion only** — the prompt tokens are masked to `-100`. This
  is the most commonly skipped detail in a fine-tuning script, and skipping it
  spends most of the gradient on text the model is always *given*.

Change `--model` to `Qwen/Qwen2.5-1.5B-Instruct` for the larger run; it fits on a
T4 with LoRA.

In [ ]:
!python src/train_lora.py --model Qwen/Qwen2.5-0.5B-Instruct --epochs 3

In [ ]:
import json
card = json.load(open("adapters/qwen05b-triage-lora/training_card.json"))
print(json.dumps(card, indent=2))
print(f"\ntrainable: {card['trainable_params']:,} of {card['total_params']:,} "
      f"({card['trainable_pct']}%)  ->  adapter is {card['adapter_bytes']/1e6:.1f} MB")

## 5. Evaluate all three conditions

The table to read is `test_unseen` — `test_seen` shares phrasing templates with
training, so it flatters the adapter.

Read the second block too ("accuracy given the output parsed"). If the adapter's
overall gain is large but its *given-parsed* accuracy is similar to the
baseline's, the win is format compliance rather than task learning. That is a
real and useful win — it is just not the same claim.

In [ ]:
!python src/evaluate.py --out evals/results.json

## 6. Sample outputs, side by side

In [ ]:
import json
results = json.load(open("evals/results.json"))

for split in ("test_seen", "test_unseen"):
    print("=" * 78)
    print(split)
    print("=" * 78)
    for condition, r in results["splits"][split].items():
        print(f"\n[{condition}] exact_match={r['exact_match']:.2f}  "
              f"json_lenient={r['json_lenient']:.2f}  prompt_tokens={r['median_prompt_tokens']}")
        for failure in r["failures"][:2]:
            print("   ticket  :", failure["ticket"][:110])
            print("   expected:", json.dumps(failure["expected"]))
            print("   got     :", failure["raw_output"][:110])
            print("   wrong   :", failure["wrong_fields"])

## 7. Push the adapter to the Hub (optional)

The adapter is a few megabytes, because only the LoRA weights are saved — the
base model is downloaded from the Hub at load time. That size difference is most
of the operational argument for LoRA: you can ship and version dozens of task
adapters against one served base model.

In [ ]:
# Needs a Hub token with write access.
# from huggingface_hub import login; login()
#
# from peft import PeftModel
# from transformers import AutoModelForCausalLM
# base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
# model = PeftModel.from_pretrained(base, "adapters/qwen05b-triage-lora")
# model.push_to_hub("<your-username>/qwen2.5-0.5b-ticket-triage-lora")
print("uncomment the lines above and run after logging in")

## 8. What to conclude, and what not to

Write down the numbers you actually got, then check them against these questions
before claiming anything:

1. **Does the adapter beat `base_fewshot`, or only `base_zeroshot`?** Beating
   zero-shot is nearly automatic for a format task and means little on its own.
2. **Is the gain in format or in task?** Compare `exact_match` against the
   `accuracy given parsed` block. A big gap means you taught it JSON, not triage.
3. **What is the `escalate` accuracy versus its majority-class baseline (~73%)?**
   That field is the only one requiring inference rather than extraction, and
   it is the one that can look fine while being worthless.
4. **How far does `test_seen` overstate `test_unseen`?** That gap is your
   template-memorisation tax, and on synthetic data it is never zero.
5. **What did it cost?** Prompt tokens, training time, adapter size. If accuracy
   is a wash, this is where the fine-tune wins — a short prompt forever versus a
   long one on every request.

Being able to answer 2 and 3 against your own result is the difference between
"I fine-tuned a model" and "I know what my fine-tune actually did".